In [16]:
# 강남구 데이터 10년치 수집함
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os
import glob

In [17]:
load_dotenv()
api_key = os.getenv("KAKAO_API_KEY")

In [18]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '건축년도', '도로명'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../../data/raw/apt_sale"
gangnam_path = "../../data/raw/apt_sale/gangnam"
# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")
# --- 여기까지 전국
# 아래부터 강남만
csv_files = glob.glob(os.path.join(gangnam_path, "*.csv"))
# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")




# 데이터프레임 병합
df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(df)}건")

읽기 완료: 아파트(매매)_실거래가_20250620103841.csv
읽기 완료: 아파트(매매)_실거래가_20250620103317.csv
읽기 완료: 아파트(매매)_실거래가_20250620103854.csv
읽기 완료: 아파트(매매)_실거래가_20250620103311.csv
읽기 완료: 아파트(매매)_실거래가_20250620103846.csv
읽기 완료: 아파트(매매)_실거래가_20250620103404.csv
읽기 완료: 아파트(매매)_실거래가_20250620103822.csv
읽기 완료: 아파트(매매)_실거래가_20250620132033.csv
읽기 완료: 아파트(매매)_실거래가_20250620132025.csv
읽기 완료: 아파트(매매)_실거래가_20250620132018.csv
읽기 완료: 아파트(매매)_실거래가_20250620103400.csv
읽기 완료: 아파트(매매)_실거래가_20250620132012.csv
읽기 완료: 아파트(매매)_실거래가_20250620100439.csv
읽기 완료: 아파트(매매)_실거래가_20250913200228.csv
읽기 완료: 아파트(매매)_실거래가_20250913200249.csv
읽기 완료: 아파트(매매)_실거래가_20250913200247.csv
읽기 완료: 아파트(매매)_실거래가_20250913200251.csv
읽기 완료: 아파트(매매)_실거래가_20250913200244.csv
읽기 완료: 아파트(매매)_실거래가_20250913200254.csv
읽기 완료: 아파트(매매)_실거래가_20250913200241.csv
읽기 완료: 아파트(매매)_실거래가_20250913200255.csv

 병합 완료: 총 555251건


# 면적당 단가 계산

In [19]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']

In [20]:
df.head()

,시군구,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원)
0,서울특별시 중랑구 면목동,면목한신,35.300,202109,1,40000,-,5,1988,중랑천로 20,1133.144476
1,서울특별시 도봉구 방학동,신동아아파트1,43.350,202109,1,45000,-,7,1988,시루봉로 107,1038.062284
2,서울특별시 성북구 길음동,길음동동부센트레빌(1278-0),114.912,202109,1,116500,-,12,2003,숭인로2길 61,1013.819270
3,서울특별시 광진구 중곡동,삼보글로벌101동,78.840,202109,1,64000,-,2,2003,긴고랑로14길 63,811.770675
4,서울특별시 광진구 구의동,구의현대2단지,84.860,202109,1,168000,-,12,1996,광나루로56길 32,1979.731322


# 아파트 나이 계산

In [21]:
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']

# 거래 순으로 나열 및 필요 없는 컬럼 삭제

In [22]:
df['구'] = df['시군구'].str.extract(r'(\S+구)')

# 계약연-월-일을 기준으로 시계열 정렬
df['계약일자'] = df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2)
df['계약일자'] = pd.to_datetime(df['계약일자'], format='%Y%m%d')

df = df.sort_values('계약일자').reset_index(drop=True)
df.drop(['시군구','계약년월','계약일','동','계약년도'], axis=1, inplace=True)

In [23]:
df = df[df['구'] == '강남구']

In [24]:
# 이상치 제거 함수 예시 (IQR 방식 등 사용자 정의 필요)
def remove_price_outliers(group):
    q1 = group['거래금액(만원)'].quantile(0.25)
    q3 = group['거래금액(만원)'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = group[(group['거래금액(만원)'] >= lower) & (group['거래금액(만원)'] <= upper)]
    return filtered

def calculate_alpha_from_age_count(age, count, N=30):
 
    raw_alpha = (1 - age / N) * np.log2(count + 1)
    alpha = max(0, min(1, raw_alpha))
    return alpha

def representative_price(prices, dates, age, N=30):

    if len(prices) == 0:
        return None  # 거래 없음 → 대표값 계산 불가

    # 가중치 alpha 계산
    count = len(prices)
    alpha = calculate_alpha_from_age_count(age, count, N)

    # 평균 거래가 (P̄)
    avg_price = np.mean(prices)

    # 최신 거래가 (P_latest)
    latest_index = np.argmax(dates)  # 거래일 기준 최대값 인덱스
    latest_price = prices[latest_index]

    # 대표 거래가 계산
    rep_price = alpha * avg_price + (1 - alpha) * latest_price
    return rep_price


def calculate_alpha_row(group, N=30):
    age = group['아파트 나이'].iloc[0]  # 해당 그룹의 아파트 나이
    count = len(group)  # 그룹 내 거래 수

    alpha = calculate_alpha_from_age_count(age, count, N)

    # 대표 row는 그룹의 첫 row 기준으로 생성
    row = group.iloc[0].copy()
    row['alpha'] = alpha
    return pd.DataFrame([row])

In [25]:
# 1. 월 단위 컬럼 생성 (이미 있다면 생략 가능)
# '계약일자' 컬럼에서 'YYYYMM' 형식의 계약년월을 다시 만듭니다.
df['계약년월'] = df['계약일자'].dt.strftime('%Y%m')

# 2. 이상치 제거 (월별 그룹 기준)
# groupby에 '계약년월'을 추가합니다.
df_filtered = df.groupby(['도로명', '단지명', '전용면적(㎡)', '계약년월'], group_keys=False)\
                .apply(remove_price_outliers)\
                .reset_index(drop=True)

# 3. 대표 row 추출 (월별 그룹 기준)
# groupby에 '계약년월'을 추가합니다.
df_monthly_representative = df_filtered.groupby(['도로명', '단지명', '전용면적(㎡)', '계약년월'], group_keys=False)\
                                     .apply(calculate_alpha_row)\
                                     .reset_index(drop=True)

# 4. 최종 정렬
df_final = df_monthly_representative.sort_values('계약일자').reset_index(drop=True)

print(f"월별 처리 후 데이터 개수: {len(df_final)}")

/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_5890/3088890929.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(remove_price_outliers)\


월별 처리 후 데이터 개수: 35613


/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_5890/3088890929.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_alpha_row)\


In [26]:
df = df_final

In [27]:
df.drop('거래금액(만원)', axis=1, inplace=True)
df = df.sort_values('계약일자').reset_index(drop=True)
df.head(1)


,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha
0,삼성동브라운스톤레전드,219.48,14,2009,선릉로 660,1047.931474,3,강남구,2012-07-02,201207,0.9


# 좌표변환

In [28]:
from pathlib import Path

OUTPUT_PATH = Path('../data/interim/apt/new/gang_nam_apt_with_long_lat.csv')  

# === 좌표 변환 === #
headers = {'Authorization': 'KakaoAK 531d049fba15b8acbb290989f6988d89'}
#Authorization: KakaoAK ${REST_API_KEY}"
def get_coords(address):
    res = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers=headers,
        params={'query': address}
    )
    if res.status_code == 200 and res.json()['documents']:
        doc = res.json()['documents'][0]
        return doc['x'], doc['y']
    return None, None

longitudes, latitudes = [], []
for address in tqdm(df['도로명'], desc="좌표 변환 중"):
    x, y = get_coords(address)
    longitudes.append(x)
    latitudes.append(y)

df['경도'] = longitudes
df['위도'] = latitudes

# === 최종 정제 및 저장 === #

df['면적당 단가(만원)'] = np.log(df['면적당 단가(만원)'])

# === 수집 못한 위도 경도는 삭제 === #
df.dropna(inplace=True)

# 디렉토리 없으면 생성
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ 저장 완료: {OUTPUT_PATH}")

좌표 변환 중: 100%|█████████████████████████████████| 35613/35613 [43:51<00:00, 13.53it/s]


✅ 저장 완료: ../data/interim/apt/new/gang_nam_apt_with_long_lat.csv


In [31]:
len(df)

34447

In [32]:
df

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,경도,위도
0,삼성동브라운스톤레전드,219.480,14,2009,선릉로 660,6.954573,3,강남구,2012-07-02,201207,0.900000,127.042332457469,37.5155511161052
1,한일,55.340,6,2001,봉은사로73길 23,6.711972,11,강남구,2012-07-02,201207,0.633333,127.049973533688,37.5139474971867
2,현대13차(208~211동),105.450,11,1983,압구정로29길 23,7.101549,29,강남구,2012-07-02,201207,0.033333,127.027364437492,37.5298193048425
3,휴먼스타빌,35.905,16,2005,도산대로 454,6.853239,7,강남구,2012-07-02,201207,0.766667,127.046056275002,37.5236847892354
4,신현대11차,183.410,2,1983,압구정로 151,6.943055,29,강남구,2012-07-03,201207,0.033333,127.024454982256,37.5267858196588
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35608,우성7,83.690,9,1987,개포로110길 15,8.150517,38,강남구,2025-06-14,202506,0.000000,127.079114775367,37.4932437695434
35609,휴먼스타빌,35.905,11,2005,도산대로 454,7.575374,20,강남구,2025-06-14,202506,0.333333,127.046056275002,37.5236847892354
35610,까치마을,49.500,12,1993,광평로19길 10,7.933036,32,강남구,2025-06-14,202506,0.000000,127.086601738647,37.485485349127
35611,쌍용플레티넘밸류,110.570,15,2007,테헤란로4길 46,7.378256,18,강남구,2025-06-16,202506,0.400000,127.03006467392,37.4955481444177
